# ADMET · Absorption assessment for rapamycin and its rapalogs

First series of ADMET-QSAR for rapamycin and rapalogs (extension of [`rapamycin_qsar.ipynb`]('rapamcyin_qsar.ipynb'); potency QSAR). This notebook focuses on **Absorption** — *can these molecules get into the body?* — for rapamycin and the five rapalogs studied in the previous notebook (sirolimus, everolimus, temsirolimus, ridaforolimus, zotarolimus).

## Purpose

Absorption answers the question of whether the drug can cross barriers and reach the bloodstream. Absorption must be investigated for every route except intravascular (IV) — and each extravascular route (oral, sublingual, buccal, rectal, transdermal, SC, IM, pulmonary, nasal, ocular) has its own absorbing barrier and its own set of absorption questions. This notebook investigates **oral absorption** of orally administered rapalogs (Sirolimus, Everolimus, Ridaforolimus) and rationalizes the choice of other administration routes for some rapalogs (Temsirolimus, Zotarolimus) using *in silico* models and measured wet-lab data.

## Conclusion

1. Openly available models to predict absorption are trained for small-molecules, and no good for rapalogs (> 900 Da macrocycles).
2. So, the **measured** data are decisive for rapalogs.
3. But, ***in silico* stack (ESOL + BOILED-Egg)** reproduces the ***qualitative*** picture — all
   practically insoluble, all outside the passive-HIA region — and gives a **defensible
   formulation-need ranking**. Quantiative predictions are way off: under-predicts solubility by ~3 log units(x1000 off).
4. For rapalogs, **limiting absorption mode = dissolution / aqueous solubility (BCS Class II)**, compounded by *measured*
   intestinal P-gp efflux and gut-wall CYP3A4 first-pass (Amidon 1995; Paine 2004; Khan 2015). Passive permeability is *not* the bottleneck for these lipophilic macrocycles.

Method choices follow accredited sources; measured drug data follow clinical
pharmacokinetic studies.

## Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors
from chembl_webresource_client.new_client import new_client


from pathlib import Path
# Notebook is stripped before push; each step's result is saved as PNG here.
RESULTS = Path("../result_rapamycin_ADMET_Absorption"); RESULTS.mkdir(exist_ok=True)
def save_fig(fig, name): fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight")
def save_img(obj, name):
    p = RESULTS / name
    obj.save(p) if hasattr(obj, "save") else p.write_bytes(obj.data)
def save_df_png(df, name, title=None):
    fig, ax = plt.subplots(figsize=(min(2 + len(df.columns) * 1.9, 16), 0.7 + 0.45 * len(df))); ax.axis("off")
    if title: ax.set_title(title, fontsize=10, loc="left")
    t = ax.table(cellText=df.astype(str).values, colLabels=list(df.columns), loc="center", cellLoc="center")
    t.auto_set_font_size(False); t.set_fontsize(8); t.scale(1, 1.4); t.auto_set_column_width(range(len(df.columns)))
    fig.savefig(RESULTS / name, dpi=150, bbox_inches="tight"); plt.close(fig)

mol_client = new_client.molecule
RAPALOGS = {
    "Sirolimus (rapamycin)": "CHEMBL413", "Everolimus": "CHEMBL1908360",
    "Temsirolimus": "CHEMBL1201182", "Ridaforolimus": "CHEMBL2103839",
    "Zotarolimus": "CHEMBL219410",
}
smiles = {n: mol_client.filter(molecule_chembl_id=c).only(["molecule_structures"])[0]
          ["molecule_structures"]["canonical_smiles"] for n, c in RAPALOGS.items()}
mols = {n: Chem.MolFromSmiles(s) for n, s in smiles.items()}
print("Loaded", len(mols), "structures.")

## Step 1 — Honest caveat

**These molecules are the hard case for *in silico* absorption.** Rapamycin and its
rapalogs are **~900–1030 Da macrocycles** — well outside the "small, drug-like" chemical
space on which solubility, permeability, and BOILED-Egg models were trained and
validated (Lipinski et al., 2001; Daina & Zoete, 2016). Any *in silico* number produced here is an **extrapolation** and must not be read as a final answer. The decisive absorption facts for these drugs come from **measured wet-lab pharmacokinetics** (Step 3). The *in silico* stack (Steps 2 and 4) is used only to **rationalize *why* absorption is poor and to rank the
formulation strategies** that fix it — not to predict absorption *de novo*.

In [ ]:
# Make the "900 Da macrocycle" point concrete
tbl = pd.DataFrame({
    "MW (Da)": {n: round(Descriptors.MolWt(m)) for n, m in mols.items()},
    "rings": {n: Descriptors.RingCount(m) for n, m in mols.items()},
    "is macrocycle (ring >= 12 atoms)": {
        n: any(len(r) >= 12 for r in m.GetRingInfo().AtomRings()) for n, m in mols.items()},
})
print(tbl.to_string())
save_df_png(tbl.reset_index().rename(columns={"index": "drug"}), "part1_macrocycle_check.png", "Part 1 - molecular size / macrocycle check")
grid = Draw.MolsToGridImage(list(mols.values()), legends=list(mols.keys()), molsPerRow=3, subImgSize=(300, 230))
save_img(grid, "part1_structures.png")
grid

## Step 2 — Designing *in silico* and measured absorption stack

### The framework and the limiting mode
Oral absorption is governed by two fundamental parameters — *solubility/dissolution*
and *permeability* — formalized by the Biopharmaceutics Classification System (BCS)
(Amidon et al., 1995). Sirolimus and its analogs are *highly lipophilic (cLogP ≈ 6) yet
practically insoluble in water*, which places them in **BCS Class II: low solubility,
high permeability**. For Class II drugs, ***the rate-limiting step of absorption is
dissolution/solubility — not membrane permeability.***

Permeability is further attenuated by two intestinal processes, both documented for
sirolimus using CYP3A4-expressing Caco-2 monolayers (Paine et al., 2004): **P-glycoprotein
efflux** and **gut-wall CYP3A4 first-pass metabolism**. Together with poor solubility, these
explain the low (~14–20%) oral bioavailability of this drug class (Khan et al., 2015).

### The stack

| Absorption mechanism | *In silico* model to use | Public data / accredited reference |
|---|---|---|
| **Aqueous & kinetic solubility → dissolution** *(limiting mode)* | **QSPR solubility** (ESOL linear model; ML on curated data) + dissolution via BCS | ESOL (Delaney, 2004); AqSolDB dataset (Sorkun et al., 2019); BCS (Amidon et al., 1995) |
| **Passive GI permeability** | **BOILED-Egg** (WLOGP vs TPSA); physicochemical rules | BOILED-Egg (Daina & Zoete, 2016); Rule of 5 (Lipinski et al., 2001); Veber et al., 2002 |

This is done in **Part 4**, *in silico*

| Absorption mechanism | measured data to use | Public data / accredited reference |
|----|---|----|
| **Active efflux (P-gp / ABCB1)** | P-gp substrate QSAR; bidirectional Caco-2 efflux ratio (A→B / B→A) | Paine et al., 2004 (Caco-2 efflux of sirolimus) |
| **Gut-wall CYP3A4 first-pass** | CYP3A4-inhibition/substrate QSAR → fraction escaping gut (Fg) | Paine et al., 2004 |

This is done in **Part 3**, P-gp and gut CYP3A4 reported from *measured* data.

## Step 3 — Measured (wet-lab) absorption data

For an out-of-domain (900Da macrocycle) class, **measured pharmacokinetics** is
the ground truth. Key findings from clinical/PK literature (*see* Part 6):

- **Sirolimus:** low average *oral bioavailability ≈ 20%*; a CYP3A4 and P-gp substrate
  with intestinal first-pass extraction demonstrated in Caco-2 monolayers — dissolution and
  intestinal efflux/metabolism jointly limit absorption (Paine et al., 2004; Khan et al., 2015).
- **Everolimus:** the 40-*O*-(2-hydroxyethyl) analog was engineered to be *more polar* to
  improve oral bioavailability over sirolimus; exposure is still P-gp/CYP3A4-variable, and
  it is marketed as an amorphous solid dispersion to overcome dissolution-limited
  absorption (Kirchner et al., 2004; Kovarik et al., 2003; Sawicki et al., 2016).
- **Temsirolimus:** given *intravenously* (ester prodrug of sirolimus) — oral delivery is
  not pursued, consistent with the class's poor absorption.
- **Ridaforolimus:** investigational; studied both IV and orally, with *oral exposure limited*
  by the same solubility/efflux barriers.
- **Zotarolimus:** engineered to be lipophilic for local elution from drug-eluting stents — *GI absorption is by design*.

In [ ]:
measured = pd.DataFrame([
    {"drug": "Sirolimus", "route": "Oral (+ IV nab)", "oral_F": "~14-20%",
     "aqueous_solubility": "practically insoluble (low ug/mL)",
     "formulation_strategy": "oral solution; nanocrystal tablet; albumin-bound IV",
     "ref": "Paine 2004; Kim 2011"},
    {"drug": "Everolimus", "route": "Oral", "oral_F": "improved vs sirolimus (variable)",
     "aqueous_solubility": "poorly soluble",
     "formulation_strategy": "amorphous solid dispersion / dispersible tablet",
     "ref": "Kirchner 2004; Kovarik 2003; Sawicki 2016"},
    {"drug": "Temsirolimus", "route": "Intravenous", "oral_F": "n/a (parenteral)",
     "aqueous_solubility": "poorly soluble",
     "formulation_strategy": "IV ester prodrug (oral not viable)", "ref": "label"},
    {"drug": "Ridaforolimus", "route": "Oral / IV (investigational)", "oral_F": "low (oral)",
     "aqueous_solubility": "poorly soluble",
     "formulation_strategy": "oral tablet (investigational)", "ref": "label"},
    {"drug": "Zotarolimus", "route": "Local (drug-eluting stent)", "oral_F": "n/a (not oral)",
     "aqueous_solubility": "poorly soluble (lipophilic by design)",
     "formulation_strategy": "polymer-coated stent (local elution)", "ref": "label"},
])
save_df_png(measured, "part3_measured_absorption_data.png", "Part 3 - measured (wet-lab) absorption data")
measured


**Summary of measured findings.** Absorption of this class is uniformly **dissolution-limited
and efflux/first-pass-attenuated**. 

The medicinal-chemistry and formulation responses map
directly onto that: increase polarity (everolimus), amorphous solid dispersions to boost
dissolution (everolimus, sirolimus nanoparticles), or bypass absorption entirely (IV for
temsirolimus; local stent elution for zotarolimus).

## Step 4 — Run the *in silico* stack to rationalize & rank the formulations

Now run the two structure-only models from Part 2 — **ESOL aqueous
solubility** (Delaney, 2004) and **BOILED-Egg/physicochemical permeability** — on the
rapalogs, and check whether they (a) correctly flag the dissolution bottleneck and
(b) rank the compounds in a way consistent with how aggressively each must be
formulated (Part 3).

In [ ]:
def esol_logS(m):
    "Delaney ESOL (2004): LogS (mol/L) from clogP, MW, rotatable bonds, aromatic proportion."
    clogp = Descriptors.MolLogP(m); mw = Descriptors.MolWt(m)
    rb = Descriptors.NumRotatableBonds(m)
    ap = sum(a.GetIsAromatic() for a in m.GetAtoms()) / m.GetNumHeavyAtoms()
    return 0.16 - 0.63 * clogp - 0.0062 * mw + 0.066 * rb - 0.74 * ap #model used for ESOL aq solubility

rows = []
for n, m in mols.items():
    logs = esol_logS(m)
    sol_ugmL = (10 ** logs) * Descriptors.MolWt(m) * 1000.0      # mol/L -> ug/mL
    tpsa, clogp = Descriptors.TPSA(m), Descriptors.MolLogP(m)
    rows.append({
        "drug": n.split(" ")[0], "MW": round(Descriptors.MolWt(m)),
        "cLogP": round(clogp, 1), "TPSA": round(tpsa),
        "ESOL_LogS(mol/L)": round(logs, 2), "ESOL_sol(ug/mL)": round(sol_ugmL, 4),
        # BOILED-Egg passive-HIA region needs TPSA <~142; flag if outside
        "passive_HIA(BOILED-Egg)": "outside (poor)" if tpsa > 142 else "inside",
    })
insilico = pd.DataFrame(rows).sort_values("ESOL_LogS(mol/L)")
save_df_png(insilico, "part4_insilico_results.png", "Part 4 - in-silico stack: ESOL + BOILED-Egg")
insilico


In [ ]:
# Compare predicted solubility to a measured anchor, and rank formulation need
MEASURED_SIROLIMUS_UGML = 2.6   # practically insoluble; literature low-ug/mL range
fig, ax = plt.subplots(figsize=(8, 4))
order = insilico.sort_values("ESOL_LogS(mol/L)")
ax.bar(order["drug"], order["ESOL_LogS(mol/L)"], color="teal")
ax.axhline(np.log10(MEASURED_SIROLIMUS_UGML / 1e6 / 914 * 1000),
           color="crimson", ls="--",
           label="measured sirolimus (~2.6 ug/mL)")
ax.set(ylabel="ESOL predicted LogS (mol/L)  [lower = less soluble]",
       title="Predicted aqueous solubility of rapalogs (ESOL) vs a measured anchor")
ax.legend(fontsize=8); plt.tight_layout()
save_fig(fig, "part4_esol_solubility.png")
plt.show()

print("Predicted solubility ranking (least -> most soluble):",
      list(order["drug"]))
print(f"\nESOL predicts ~{order['ESOL_sol(ug/mL)'].iloc[0]:.4f}-"
      f"{order['ESOL_sol(ug/mL)'].iloc[-1]:.4f} ug/mL — i.e. 'practically insoluble' for all.")
print("Measured sirolimus solubility is ~2.6 ug/mL: ESOL UNDER-predicts by ~3 orders of "
      "magnitude (molecule size out-of-domain penalty).")

### Conclusion

*In silico* modeling with rapalogs did not work due to low **domain applicability**. But, *in silico* models ...

- **Correctly identified the limiting mode.** ESOL puts every rapalog at LogS ≈ −9 to −10
  (sub-µg/mL) and BOILED-Egg places them all outside the passive-absorption region
  (TPSA ≈ 195–242 ≫ ~142). Both models flag at **dissolution/solubility**, exactly the BCS-II
  bottleneck seen in the lab.
  
- **Gave a defensible ranking.** The most lipophilic analog (ridaforolimus, cLogP ≈ 7.7) is predicted to be the least soluble → the one expected to need the most aggressive solubility-enabling formulation. On the other hand, the more polar everolimus ranks better, consistent with the analog's engineering rationale for improved oral exposure.

- **Were quantitatively off.** ESOL under-predicts sirolimus's measured ~2.6 µg/mL by ~1000×. For 900-Da macrocycles, the *number* is not trustworthy — only the *direction* and *relative order* are, and only as a formulation-triage aid.

## Citations

Bibliographic metadata retrieved from **PubMed**.

**Drug pharmacokinetics / absorption**

1. Paine, M. F.; Leung, L. Y.; Watkins, P. B. New Insights into Drug Absorption: Studies with
   Sirolimus. *Ther. Drug Monit.* **2004**, *26* (5), 463–467.
   https://doi.org/10.1097/00007691-200410000-00001.
2. Kirchner, G. I.; Meier-Wiedenbach, I.; Manns, M. P. Clinical Pharmacokinetics of Everolimus.
   *Clin. Pharmacokinet.* **2004**, *43* (2), 83–95.
   https://doi.org/10.2165/00003088-200443020-00002.
3. Kovarik, J. M.; Noe, A.; Berthier, S.; McMahon, L.; Langholff, W. K.; Marion, A. S.;
   Hoyer, P. F.; Ettenger, R.; Rordorf, C. Clinical Development of an Everolimus Pediatric
   Formulation: Relative Bioavailability, Food Effect, and Steady-State Pharmacokinetics.
   *J. Clin. Pharmacol.* **2003**, *43* (2), 141–147.
   https://doi.org/10.1177/0091270002239822.
4. Khan, S.; Khan, S.; Baboota, S.; Ali, J. Immunosuppressive Drug Therapy — Biopharmaceutical
   Challenges and Remedies. *Expert Opin. Drug Deliv.* **2015**, *12* (8), 1333–1349.
   https://doi.org/10.1517/17425247.2015.1005072.
5. Sawicki, E.; Schellens, J. H. M.; Beijnen, J. H.; Nuijen, B. Inventory of Oral Anticancer
   Agents: Pharmaceutical Formulation Aspects with Focus on the Solid Dispersion Technique.
   *Cancer Treat. Rev.* **2016**, *50*, 247–263. https://doi.org/10.1016/j.ctrv.2016.09.012.
6. Kim, M.-S.; Kim, J.-S.; Park, H. J.; Cho, W. K.; Cha, K.-H.; Hwang, S.-J. Enhanced
   Bioavailability of Sirolimus via Preparation of Solid Dispersion Nanoparticles Using a
   Supercritical Antisolvent Process. *Int. J. Nanomedicine* **2011**, *6*, 2997–3009.
   https://doi.org/10.2147/IJN.S26546.

***In silico* methods & datasets**

7. Amidon, G. L.; Lennernäs, H.; Shah, V. P.; Crison, J. R. A Theoretical Basis for a
   Biopharmaceutic Drug Classification: The Correlation of in Vitro Drug Product Dissolution
   and in Vivo Bioavailability. *Pharm. Res.* **1995**, *12* (3), 413–420.
   https://doi.org/10.1023/A:1016212804288.
8. Lipinski, C. A.; Lombardo, F.; Dominy, B. W.; Feeney, P. J. Experimental and Computational
   Approaches to Estimate Solubility and Permeability in Drug Discovery and Development
   Settings. *Adv. Drug Deliv. Rev.* **2001**, *46* (1–3), 3–26.
   https://doi.org/10.1016/S0169-409X(00)00129-0.
9. Veber, D. F.; Johnson, S. R.; Cheng, H.-Y.; Smith, B. R.; Ward, K. W.; Kopple, K. D.
   Molecular Properties That Influence the Oral Bioavailability of Drug Candidates.
   *J. Med. Chem.* **2002**, *45* (12), 2615–2623. https://doi.org/10.1021/jm020017n.
10. Delaney, J. S. ESOL: Estimating Aqueous Solubility Directly from Molecular Structure.
    *J. Chem. Inf. Comput. Sci.* **2004**, *44* (3), 1000–1005.
    https://doi.org/10.1021/ci034243x.
11. Sorkun, M. C.; Khetan, A.; Er, S. AqSolDB, a Curated Reference Set of Aqueous Solubility
    and 2D Descriptors for a Diverse Set of Compounds. *Sci. Data* **2019**, *6* (1), 143.
    https://doi.org/10.1038/s41597-019-0151-1.
12. Daina, A.; Zoete, V. A BOILED-Egg to Predict Gastrointestinal Absorption and Brain
    Penetration of Small Molecules. *ChemMedChem* **2016**, *11* (11), 1117–1121.
    https://doi.org/10.1002/cmdc.201600182.
13. Daina, A.; Michielin, O.; Zoete, V. SwissADME: A Free Web Tool to Evaluate
    Pharmacokinetics, Drug-Likeness and Medicinal Chemistry Friendliness of Small Molecules.
    *Sci. Rep.* **2017**, *7*, 42717. https://doi.org/10.1038/srep42717.

*Attribution: bibliographic data retrieved from PubMed. Routes of administration for
temsirolimus, ridaforolimus, and zotarolimus reflect approved/registered product labels.*

## Addendum — Applicability-domain check of the absorption ML models (Tanimoto)

**Purpose** Check how off the rapalogs are for three public absorption benchmarks.

**Methodology.** A QSAR/ML model is only reliable for molecules resembling those it was trained on.
A simple, widely used measure of that is the **Tanimoto coefficient** (Morgan-fingerprint similarity)
of the query to its **nearest neighbour in the training set**: higher similarity → more reliable
prediction (Sheridan et al., 2004). Below, for each rapalog, I computed its maximum Tanimoto to
the training sets of three public absorption benchmarks — **HIA**, **oral bioavailability**, and
**Caco-2 permeability** (Therapeutics Data Commons; Huang et al., 2022) — and also test exact compound
membership by InChIKey.

In [ ]:
# Applicability domain = similarity of each rapalog to each model's TRAINING set
# (max Tanimoto on Morgan fingerprints; Sheridan et al. 2004). No model is trained here -
# the point is whether these models are even usable for 900 Da macrocycles.
import zipfile, requests
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator

ZIP = Path("../data/tdc_admet_group.zip")
if not ZIP.exists():
    ZIP.write_bytes(requests.get("https://dataverse.harvard.edu/api/access/datafile/4426004",
                                 headers={"User-Agent": "Mozilla/5.0"}, timeout=120).content)
zf = zipfile.ZipFile(ZIP)
_GEN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

DATASETS = {"HIA": "hia_hou", "Oral bioavailability": "bioavailability_ma", "Caco-2": "caco2_wang"}
train = {}
for label, ds in DATASETS.items():
    d = pd.concat([pd.read_csv(zf.open(f"admet_group/{ds}/{f}")) for f in ("train_val.csv", "test.csv")], ignore_index=True)
    fps, keys = [], set()
    for s in d["Drug"]:
        m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        if m:
            fps.append(_GEN.GetFingerprint(m)); keys.add(Chem.MolToInchiKey(m))
    train[label] = {"fps": fps, "keys": keys, "n": len(fps)}
    print(f"{label:20s} training compounds = {len(fps)}")

rap_bv = {n: _GEN.GetFingerprint(m) for n, m in mols.items()}
rap_key = {n: Chem.MolToInchiKey(m) for n, m in mols.items()}
rows = []
for n, bv in rap_bv.items():
    r = {"rapalog": n.split(" ")[0]}
    for label in DATASETS:
        r[f"{label} maxT"] = round(max(DataStructs.BulkTanimotoSimilarity(bv, train[label]["fps"])), 2)
        r[f"{label} in-set"] = "yes" if rap_key[n] in train[label]["keys"] else "no"
    rows.append(r)
ad = pd.DataFrame(rows)
save_df_png(ad, "addendum_applicability_domain_table.png",
            "Addendum - applicability domain: max Tanimoto of each rapalog to each absorption model's training set")
ad

In [ ]:
# Visualize the applicability domain against a common reliability threshold
fig, ax = plt.subplots(figsize=(9, 4)); x = np.arange(len(ad)); w = 0.25
for i, label in enumerate(DATASETS):
    ax.bar(x + (i - 1) * w, ad[f"{label} maxT"], w, label=label)
ax.axhline(0.3, color="red", ls="--", label="in-domain threshold ~0.3 (Sheridan 2004)")
ax.set_xticks(x); ax.set_xticklabels(ad["rapalog"], rotation=20)
ax.set(ylabel="max Tanimoto to training set", ylim=(0, 1.05),
       title="Applicability domain of the absorption ML models for the rapalogs")
ax.legend(fontsize=8); plt.tight_layout()
save_fig(fig, "addendum_applicability_domain.png"); plt.show()

### What the applicability-domain check shows

- **HIA and oral bioavailability — out of domain.** Every rapalog's nearest training neighbour is only
  ~0.2–0.4 Tanimoto away, well below the ~0.3 reliability guideline (Sheridan et al., 2004). *None
  of the rapalogs is in either training set* (InChIKey check). Any HIA/bioavailability prediction for
  them is therefore an **extrapolation**.
- **Caco-2 — seemingly high, but a false alarm.** Sirolimus shows Tanimoto = 1.0 because it is *in
  the Caco-2 training set* (confirmed by InChIKey); the other rapalogs sit at ~0.83–0.87 only because
  sirolimus is their nearest neighbour. So a high Caco-2 "domain" score here reflects **memorization**
  (sirolimus) or **near-analogue recall** (the rest), not genuine coverage of macrocycle space.

**Conclusion.** By the Tanimoto applicability-domain test, the standard ML absorption models are **not
applicable to the rapalogs**: they are 900 Da macrocycles outside the models' small-molecule training
space.

### Addendum citations (ACS style)

Bibliographic metadata retrieved from **PubMed**.

1. Sheridan, R. P.; Feuston, B. P.; Maiorov, V. N.; Kearsley, S. K. Similarity to Molecules in the
   Training Set Is a Good Discriminator for Prediction Accuracy in QSAR. *J. Chem. Inf. Comput. Sci.*
   **2004**, *44* (6), 1912–1928. https://doi.org/10.1021/ci049782w.
2. Huang, K.; Fu, T.; Gao, W.; Zhao, Y.; Roohani, Y.; Leskovec, J.; Coley, C. W.; Xiao, C.; Sun, J.;
   Zitnik, M. Artificial Intelligence Foundation for Therapeutic Science. *Nat. Chem. Biol.* **2022**,
   *18* (10), 1033–1036. https://doi.org/10.1038/s41589-022-01131-2.
3. Fagerholm, U.; Hellberg, S.; Alvarsson, J.; Spjuth, O. In Silico Predictions of the Gastrointestinal
   Uptake of Macrocycles in Man Using Conformal Prediction Methodology. *J. Pharm. Sci.* **2022**,
   *111* (9), 2614–2619. https://doi.org/10.1016/j.xphs.2022.05.010.

*Attribution: PubMed. Training-set similarity computed against the TDC `admet_group` benchmark
(hia_hou, bioavailability_ma, caco2_wang).*